# Project Besar Analitik dan Visualisasi Data: Data Cleaning

Tahap pertama dalam pipeline data science adalah **Data Cleaning**.

Pada tahap ini kita akan:
1. Memahami struktur dataset mentah
2. Mengidentifikasi dan menangani missing value
3. Mengecek dan menangani data duplikat
4. Melakukan validasi tipe data
5. Mendeteksi dan menangani anomali data
6. Menyimpan dataset bersih untuk tahap selanjutnya


## 1. Import Library


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


## 2. Load Dataset Mentah

Memuat dataset Transjakarta dari file CSV.


In [ ]:
# Load dataset mentah
df = pd.read_csv('dfTransjakarta.csv')

print("=" * 50)
print("       INFORMASI DATASET MENTAH")
print("=" * 50)
print(f"Jumlah Baris  : {df.shape[0]:,}")
print(f"Jumlah Kolom  : {df.shape[1]}")
print(f"Memory Usage  : {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("=" * 50)
print()
display(df.head())


## 3. Eksplorasi Struktur Data

Melihat tipe data dan informasi kolom sebelum melakukan cleaning.


In [ ]:
# Informasi tipe data dan non-null count
df.info()


In [ ]:
# Statistik deskriptif untuk kolom numerik
display(df.describe())


In [ ]:
# Daftar seluruh kolom beserta tipe datanya
col_info = pd.DataFrame({
    'Tipe Data': df.dtypes,
    'Non-Null Count': df.notnull().sum(),
    'Null Count': df.isnull().sum(),
    'Unique Values': df.nunique()
})
display(col_info)


## 4. Analisis Missing Value

Mengidentifikasi kolom mana saja yang memiliki missing value beserta proporsinya.


In [ ]:
# Menghitung jumlah dan persentase missing value per kolom
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Jumlah Missing': missing,
    'Persentase (%)': missing_pct.round(2)
})

# Tampilkan hanya kolom yang memiliki missing value
missing_df = missing_df[missing_df['Jumlah Missing'] > 0].sort_values(
    by='Jumlah Missing', ascending=False
)

print(f"Total missing value : {df.isnull().sum().sum():,}")
print(f"Kolom dengan missing: {len(missing_df)} dari {len(df.columns)} kolom")
print()
display(missing_df)


### Interpretasi Missing Value

Berdasarkan hasil analisis di atas:

| Kolom | Missing | Kemungkinan Penyebab |
|-------|---------|---------------------|
| `tapOutStops` | Tinggi | Penumpang tidak tap out |
| `corridorName` | Tinggi | Data koridor tidak tercatat |
| `tapOutStopsName` | Sedang | Terkait dengan tapOutStops yang kosong |
| `tapOutStopsLat/Lon` | Sedang | Lokasi tap out tidak tersedia |
| `stopEndSeq` | Sedang | Urutan halte akhir tidak tersedia |
| `tapOutTime` | Sedang | Waktu tap out tidak tercatat |
| `corridorID` | Sedang | ID koridor tidak tercatat |
| `tapInStops` | Sedang | ID halte masuk tidak tercatat |
| `payAmount` | Rendah | Tarif tidak tercatat |


## 5. Penanganan Missing Value

**Strategi: Drop Row** (menghapus baris yang mengandung missing value).

**Alasan:**
- Dataset cukup besar (37.900 baris) sehingga kehilangan beberapa baris tidak akan mengubah distribusi data secara signifikan
- Baris dengan missing value pada kolom kunci (corridorName, tapOutTime, payAmount) tidak dapat dihitung durasi perjalanannya sehingga tidak berguna untuk clustering
- Mengisi data dengan mean/median/mode dapat menimbulkan bias pada hasil clustering K-Means
- Menjaga integritas dan keaslian data untuk analisis yang lebih akurat


In [ ]:
# Simpan jumlah baris sebelum cleaning
rows_before = len(df)

# Menghapus baris yang mengandung missing value
df_clean = df.dropna().copy()

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print("=" * 50)
print("     HASIL PENANGANAN MISSING VALUE")
print("=" * 50)
print(f"Baris sebelum cleaning : {rows_before:,}")
print(f"Baris setelah cleaning  : {rows_after:,}")
print(f"Baris yang dihapus      : {rows_removed:,}")
print(f"Persentase data tersisa : {(rows_after/rows_before*100):.2f}%")
print("=" * 50)


In [ ]:
# Verifikasi: pastikan tidak ada missing value tersisa
remaining_missing = df_clean.isnull().sum().sum()
print(f"Total missing value tersisa: {remaining_missing}")

if remaining_missing == 0:
    print("✅ Semua missing value berhasil ditangani.")
else:
    print("❌ Masih terdapat missing value!")


## 6. Pengecekan Data Duplikat

Mengecek apakah terdapat baris duplikat pada dataset.


In [ ]:
# Cek jumlah baris duplikat
duplicates = df_clean.duplicated().sum()
print(f"Jumlah baris duplikat: {duplicates:,}")

if duplicates > 0:
    print(f"Menghapus {duplicates:,} baris duplikat...")
    df_clean = df_clean.drop_duplicates()
    print(f"Baris setelah penghapusan duplikat: {len(df_clean):,}")
else:
    print("✅ Tidak ada baris duplikat pada dataset.")


## 7. Validasi Tipe Data

Memastikan kolom-kolom tertentu memiliki tipe data yang sesuai.


In [ ]:
# Konversi tipe data yang diperlukan
print("--- Validasi Tipe Data ---")
print()

# 1. tapInTime dan tapOutTime harus datetime
print(f"tapInTime  (sebelum) : {df_clean['tapInTime'].dtype}")
print(f"tapOutTime (sebelum) : {df_clean['tapOutTime'].dtype}")

df_clean['tapInTime'] = pd.to_datetime(df_clean['tapInTime'])
df_clean['tapOutTime'] = pd.to_datetime(df_clean['tapOutTime'])

print(f"tapInTime  (sesudah) : {df_clean['tapInTime'].dtype}")
print(f"tapOutTime (sesudah) : {df_clean['tapOutTime'].dtype}")
print()

# 2. direction harus integer (0 atau 1)
print(f"direction  (sebelum) : {df_clean['direction'].dtype}")
df_clean['direction'] = df_clean['direction'].astype(int)
print(f"direction  (sesudah) : {df_clean['direction'].dtype}")
print(f"direction values     : {df_clean['direction'].unique()}")
print()

# 3. stopEndSeq harus integer
print(f"stopEndSeq (sebelum) : {df_clean['stopEndSeq'].dtype}")
df_clean['stopEndSeq'] = df_clean['stopEndSeq'].astype(int)
print(f"stopEndSeq (sesudah) : {df_clean['stopEndSeq'].dtype}")
print()

# 4. payAmount harus integer
print(f"payAmount  (sebelum) : {df_clean['payAmount'].dtype}")
df_clean['payAmount'] = df_clean['payAmount'].astype(int)
print(f"payAmount  (sesudah) : {df_clean['payAmount'].dtype}")

print()
print("✅ Validasi tipe data selesai.")


## 8. Deteksi Anomali Data

Mengecek apakah terdapat nilai-nilai yang tidak masuk akal secara logika bisnis.


In [ ]:
# 1. Cek payCardBirthDate (tahun lahir)
print("--- Validasi Tahun Lahir (payCardBirthDate) ---")
print(f"Min : {df_clean['payCardBirthDate'].min()}")
print(f"Max : {df_clean['payCardBirthDate'].max()}")
print(f"Mean: {df_clean['payCardBirthDate'].mean():.0f}")
print()

# 2. Cek payCardSex (jenis kelamin)
print("--- Distribusi Jenis Kelamin (payCardSex) ---")
print(df_clean['payCardSex'].value_counts())
print()

# 3. Cek payAmount
print("--- Validasi payAmount ---")
print(f"Min     : Rp {df_clean['payAmount'].min():,}")
print(f"Max     : Rp {df_clean['payAmount'].max():,}")
print(f"Mean    : Rp {df_clean['payAmount'].mean():,.0f}")
print(f"Nol (0) : {(df_clean['payAmount'] == 0).sum():,} baris")
print(f"Negatif : {(df_clean['payAmount'] < 0).sum():,} baris")
print()

# 4. Cek urutan halte
print("--- Validasi Urutan Halte ---")
print(f"stopStartSeq: min={df_clean['stopStartSeq'].min()}, max={df_clean['stopStartSeq'].max()}")
print(f"stopEndSeq  : min={df_clean['stopEndSeq'].min()}, max={df_clean['stopEndSeq'].max()}")
print()

# 5. Cek apakah tapOutTime selalu >= tapInTime
invalid_time = df_clean[df_clean['tapOutTime'] < df_clean['tapInTime']]
print(f"tapOutTime < tapInTime (durasi negatif): {len(invalid_time):,} baris")

if len(invalid_time) > 0:
    print("⚠️ Terdapat anomali waktu. Baris ini akan dihapus di tahap Feature Engineering.")
else:
    print("✅ Tidak ada anomali waktu.")


### Catatan tentang payAmount = 0

Baris dengan `payAmount = 0` **tidak dihapus** karena:
- Transjakarta memiliki program **gratis ongkos** untuk beberapa rute tertentu
- Penumpang pelajar dan lansia sering mendapat **subsidi penuh**
- Data ini valid secara bisnis dan **relevan untuk analisis segmentasi**
- Menghapus data ini akan menghilangkan hampir 44% dataset


## 9. Ringkasan Data Cleaning

Perbandingan dataset sebelum dan sesudah proses cleaning.


In [ ]:
print("=" * 55)
print("         RINGKASAN DATA CLEANING")
print("=" * 55)
print(f"{'Metrik':<30} {'Sebelum':>10} {'Sesudah':>10}")
print("-" * 55)
print(f"{'Jumlah Baris':<30} {df.shape[0]:>10,} {df_clean.shape[0]:>10,}")
print(f"{'Jumlah Kolom':<30} {df.shape[1]:>10} {df_clean.shape[1]:>10}")
print(f"{'Total Missing Value':<30} {df.isnull().sum().sum():>10,} {df_clean.isnull().sum().sum():>10,}")
print(f"{'Total Duplikat':<30} {df.duplicated().sum():>10,} {df_clean.duplicated().sum():>10,}")
print(f"{'Baris Dihapus':<30} {'':<10} {df.shape[0] - df_clean.shape[0]:>10,}")
print(f"{'Data Tersisa (%)':<30} {'':<10} {(df_clean.shape[0]/df.shape[0]*100):>9.2f}%")
print("=" * 55)
print()

print("Teknik yang digunakan:")
print("  1. Drop Row untuk Missing Value")
print("  2. Drop Duplicates")
print("  3. Konversi Tipe Data (datetime, int)")
print("  4. Validasi Anomali (tahun lahir, durasi, payAmount)")
print()
display(df_clean.head(10))


In [ ]:
# Verifikasi akhir
print("--- Tipe Data Final ---")
print(df_clean.dtypes)
print()
print(f"Missing Value : {df_clean.isnull().sum().sum()}")
print(f"Duplikat      : {df_clean.duplicated().sum()}")
df_clean.info()


## 10. Export Dataset Bersih

Menyimpan dataset yang sudah dibersihkan ke file CSV baru.

File output: **`dfTransjakarta_clean.csv`**


In [ ]:
# Export dataset bersih
output_path = 'dfTransjakarta_clean.csv'
df_clean.to_csv(output_path, index=False)

print("=" * 50)
print("       DATA CLEANING SELESAI")
print("=" * 50)
print(f"File Output : {output_path}")
print(f"Ukuran      : {df_clean.shape[0]:,} baris x {df_clean.shape[1]} kolom")
print("=" * 50)
print()
print("Dataset siap untuk tahap selanjutnya: Feature Engineering.")
